<a href="https://colab.research.google.com/github/thenameisRICKKU/IPL-Win-Probability-Analyser/blob/main/IPL_Win_Probability_Analyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# dependencies
!pip install scikit-learn pandas -q

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

In [4]:
# Load dataset
url = 'https://raw.githubusercontent.com/thenameisRICKKU/IPL-Win-Probability-Analyser/main/IPLLast5YearData.zip'
df = pd.read_csv(url, compression='zip')

# Dynamically filter for the modern T20 era (rolling 5-year inclusive window)
latest_year = df['year'].max()
df = df[df['year'] >= (latest_year - 4)]

# Isolate 2nd innings run chases with valid results
chase_df = df[(df['innings'] == 2) & df['runs_target'].notna() & df['match_won_by'].notna()].copy()

# Base Feature Engineering: Match Situation
chase_df['balls_left'] = (120 - chase_df['team_balls']).clip(lower=1)
chase_df['safe_team_balls'] = chase_df['team_balls'].clip(lower=1)
chase_df['runs_left'] = chase_df['runs_target'] - chase_df['team_runs']
chase_df['wickets_left'] = 10 - chase_df['team_wicket']

# Advanced Feature Engineering: Run Rates & Era Dynamics
chase_df['crr'] = (chase_df['team_runs'] / chase_df['safe_team_balls']) * 6
chase_df['rrr'] = (chase_df['runs_left'] / chase_df['balls_left']) * 6
chase_df.replace([np.inf, -np.inf], 99, inplace=True)

chase_df['is_death_overs'] = (chase_df['balls_left'] <= 30).astype(int)
chase_df['wickets_x_balls'] = chase_df['wickets_left'] * chase_df['balls_left']

# Define Target Variable
chase_df['chasing_team_won'] = (chase_df['batting_team'] == chase_df['match_won_by']).astype(int)

# Finalize training dataset
features = ['runs_target', 'runs_left', 'balls_left', 'wickets_left', 'crr', 'rrr', 'is_death_overs', 'wickets_x_balls']
final_data = chase_df[features + ['chasing_team_won']].dropna()

In [5]:
# Isolate features and target
X = final_data[features]
y = final_data['chasing_team_won']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (Critical for Logistic Regression probability calibration)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the Logistic Regression engine
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Evaluate model performance
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Training Complete.")
print(f"Rolling 5-Year Era Accuracy: {accuracy * 100:.2f}%")

Model Training Complete.
Rolling 5-Year Era Accuracy: 80.45%


In [6]:
def predict_win_probability(target_score, current_score, wickets_lost, overs_completed):
    """
    Calculates the real-time, calibrated win probability for an active IPL run chase.

    Parameters:
    target_score (int): Runs required to win the match.
    current_score (int): Current runs scored by the chasing team.
    wickets_lost (int): Number of wickets fallen.
    overs_completed (float): Overs bowled (e.g., 19.3 for 19 overs and 3 balls).
    """
    # Parse match state
    runs_left = target_score - current_score
    overs = int(overs_completed)
    balls_completed = max(1, (overs * 6) + round((overs_completed - overs) * 10))
    balls_left = max(1, 120 - balls_completed)
    wickets_left = 10 - wickets_lost

    # Calculate rolling metrics
    crr = (current_score / balls_completed) * 6
    rrr = (runs_left / balls_left) * 6
    is_death_overs = 1 if balls_left <= 30 else 0
    wickets_x_balls = wickets_left * balls_left

    # Construct input vector and scale
    input_data = pd.DataFrame([[
        target_score, runs_left, balls_left, wickets_left,
        crr, rrr, is_death_overs, wickets_x_balls
    ]], columns=features)

    input_scaled = scaler.transform(input_data)

    # Generate probabilities
    prob = model.predict_proba(input_scaled)[0]

    # Broadcast Output
    print("=" * 45)
    print(" 📺 LIVE IPL WIN PROBABILITY ANALYZER 📺 ")
    print("=" * 45)
    print(f"Target: {target_score} | Score: {current_score}/{wickets_lost} ({overs_completed} Ov)")
    print(f"Need {runs_left} runs from {balls_left} balls.")
    print(f"CRR: {crr:.2f} | RRR: {rrr:.2f}")
    print("-" * 45)
    print(f"Batting Team (Chasing):  {prob[1] * 100:.1f}%")
    print(f"Bowling Team (Defending): {prob[0] * 100:.1f}%")
    print("=" * 45)

# --- STRESS TEST ---
# Simulating a modern high-pressure chase: 20 runs needed off 6 balls, 5 wickets in hand.
predict_win_probability(target_score=229, current_score=209, wickets_lost=5, overs_completed=19)

 📺 LIVE IPL WIN PROBABILITY ANALYZER 📺 
Target: 229 | Score: 209/5 (19 Ov)
Need 20 runs from 6 balls.
CRR: 11.00 | RRR: 20.00
---------------------------------------------
Batting Team (Chasing):  10.2%
Bowling Team (Defending): 89.8%
